# Movie Sentiments Dashboard

In [994]:
import subprocess, sys
for pkg in ['dash', 'dash-bootstrap-components', 'plotly', 'pandas', 'nbformat']:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
    
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

print("Packages installed")

Packages installed


In [995]:
# ── Data ──────────────────────────────────────────────────────

movies_df = pd.read_csv('tmdb_movie_enriched_scl_aggregate.csv')

movies_df['release_date'] = pd.to_datetime(movies_df['release_date'], errors='coerce')
movies_df['year'] = movies_df['release_date'].dt.year
movies_df['revenue'] = pd.to_numeric(movies_df['revenue'], errors='coerce')
movies_df['sentiment_mean'] = pd.to_numeric(movies_df['sentiment_mean'], errors='coerce')

# Movie Data Filter --------------------------------------------------------------------
movies_df = movies_df[
    movies_df['year'].between(1910, 2025) &
    (movies_df['revenue'] > 0) &
    (movies_df['vote_count'] >= 10) &
    movies_df['dominant_polarity'].notna()
].copy()

movies_df['sentiment_category'] = movies_df['sentiment_mean'].apply(
    lambda x: 'positive' if x >= 0 else 'negative'
)

# Keywords -------------------------------------------------------
keywords_df = pd.read_csv('tmdb_movie_keyword_pairs_scl_enriched.csv')
keywords_df = keywords_df[
    (keywords_df['is_narrative'] == True) &
    (keywords_df['keyword_type'] != 'location')
].copy()

print(f'{len(movies_df):,} movies, {len(keywords_df):,} narrative keyword pairs')
movies_df['sentiment_category'].value_counts()

11,541 movies, 728,027 narrative keyword pairs


sentiment_category
positive    5908
negative    5633
Name: count, dtype: int64

In [996]:
# Theme & Colors --------------------------------------------------------------------
COLORS = {
    'positive': 'rgb(255, 122, 48)',
    'negative': 'rgb(61, 116, 182)',
    'neutral': '#D6D4CC',
    'bg': '#DDDDDD',
    'bg-light': '#f3f6f4',
    'text': '#36454F',
    'grid': 'rgba(0,0,0,0.08)',
}

layout_base = dict(
    plot_bgcolor=COLORS['bg-light'],
    paper_bgcolor=COLORS['bg'],
    font=dict(family='Inter, sans-serif', size=13, color=COLORS['text']),
    hovermode='x unified',
    showlegend=False,
    margin=dict(l=80, r=40, t=100, b=60),
)

In [997]:
def make_area_chart(df, sentiments=['positive', 'negative'], title='Positive Films Earn More at the Box Office'):
    data = (df.groupby(['year', 'sentiment_category'])['revenue']
              .sum()
              .unstack(fill_value=0)
              .reindex(columns=sentiments, fill_value=0))
    smoothed = data.rolling(5, min_periods=1, center=True).mean()
    stack_order = ['negative', 'positive']
    patterns = {'positive': '+', 'negative': '/'}
    fig = go.Figure()
    for s in stack_order:
        if s not in sentiments:
            continue
        fig.add_trace(go.Scatter(
            x=smoothed.index, y=smoothed[s],
            name=s.capitalize(),
            hoverinfo='x+y',
            mode='lines',
            line=dict(width=0.5, color=COLORS[s]),
            fillpattern=dict(
                shape=patterns[s],
                fgcolor=COLORS[s],
                fillmode='overlay',
                size=6,
                solidity=0.2,
            ),
            stackgroup='one',
            groupnorm='percent',
        ))

    fig.update_layout(title=dict(text=title, x=0.5, font=dict(size=20, color=COLORS['text'])),
                      **layout_base, height=600)
    fig.add_annotation(text='Positive', xref='paper', yref='paper',
                        x=0.95, y=0.85, showarrow=False,
                        font=dict(size=14, color=COLORS['text']))
    fig.add_annotation(text='Negative', xref='paper', yref='paper',
                        x=0.95, y=0.15, showarrow=False,
                        font=dict(size=14, color=COLORS['text']))

    fig.update_xaxes(title='Year', showgrid=True, gridcolor=COLORS['grid'], zeroline=False)
    fig.update_yaxes(title='% of Total Revenue', range=[0, 100], ticksuffix='%')
    return fig
fig = make_area_chart(movies_df)
fig.show()

In [998]:
NUM_GENRES = 6

def make_genre_chart(df, year_range):
    exploded = df.assign(genre=df['genres'].str.split(', ')).explode('genre')
    top_genres = exploded['genre'].value_counts().head(NUM_GENRES).index.tolist()
    exploded = exploded[exploded['genre'].isin(top_genres)]

    counts = exploded.groupby(['genre', 'sentiment_category']).size().unstack(fill_value=0)
    counts = counts.reindex(columns=['positive', 'negative'], fill_value=0)
    totals = counts.sum(axis=1)
    pcts = counts.div(totals, axis=0) * 100

    pcts = pcts.loc[top_genres]
    counts = counts.loc[top_genres]

    pcts['negative'] = pcts['negative'] * -1

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=pcts.index, x=pcts['negative'],
        name='Negative', orientation='h',
        marker=dict(color=COLORS['negative'], opacity=0.9),
        text=[f"{c} ({p:.0f}%)" for c, p in zip(counts['negative'], counts['negative'] / totals.loc[top_genres] * 100)],
        textposition='inside',
        hoverinfo='skip',
        legendrank=1,
    ))
    fig.add_trace(go.Bar(
        y=pcts.index, x=pcts['positive'],
        name='Positive', orientation='h',
        marker=dict(color=COLORS['positive'], opacity=0.9),
        text=[f"{c} ({p:.0f}%)" for c, p in zip(counts['positive'], counts['positive'] / totals.loc[top_genres] * 100)],
        textposition='inside',
        hoverinfo='skip',
        legendrank=2,
    ))

    title = f"Genre Tug-of-War from {year_range[0]}-{year_range[1]}"
    fig.update_layout(**layout_base)
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=18, color=COLORS['text'])),
        barmode='relative',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='center', x=0.5),
        yaxis=dict(autorange='reversed', automargin=True),
        margin=dict(l=0, r=0, t=60, b=30),
        height=500,
    )
    fig.update_xaxes(
        zeroline=True,
        zerolinecolor='black',
        zerolinewidth=1,
        tickmode='array',
        tickvals=[-100, -50, -25, 0, 25,50, 100],
        ticktext=['100%', '25%', '50%', '0', '25%', '50%', '100%'],
        showgrid=False,
    )
    return fig

fig = make_genre_chart(movies_df, [1936, 2023])
fig.show()
# diverging bar chart https://plotly.com/python/horizontal-bar-charts/

In [999]:
NUM_KEYWORDS = 10

def make_keyword_chart(df, year_range):
    movie_ids = df['tmdb_movie_id']
    kw = keywords_df[keywords_df['tmdb_movie_id'].isin(movie_ids)]
    kw = kw[kw['scl_polarity'].isin(['positive', 'negative'])]

    top_kw = kw['name'].value_counts().head(NUM_KEYWORDS)
    top_names = top_kw.index.tolist()
    top_counts = top_kw.values

    kw_top = kw[kw['name'].isin(top_names)]
    polarity_map = kw_top.drop_duplicates('name').set_index('name')['scl_polarity']
    bar_colors = [COLORS.get(polarity_map.get(n, 'neutral'), COLORS['neutral']) for n in top_names]

    fig = go.Figure()
    fig.add_trace(go.Bar(
        y=top_names,
        x=top_counts,
        orientation='h',
        marker=dict(color=bar_colors, opacity=0.9),
        text=top_counts,
        textposition='outside',
        hoverinfo='skip',
    ))

    title = f"Top Keywords from {year_range[0]}-{year_range[1]}"
    fig.update_layout(**layout_base)
    fig.update_layout(
        title=dict(text=title, x=0.5, font=dict(size=18, color=COLORS['text'])),
        yaxis=dict(autorange='reversed', automargin=True),
        xaxis=dict(title='Movie Count', showgrid=True, gridcolor=COLORS['grid']),
        margin=dict(l=0, r=30, t=60, b=30),
        height=500,
    )
    return fig

fig = make_keyword_chart(movies_df, [1952, 1963])
fig.show()

In [1000]:
# Layout --------------------------------------------------------------------
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc

MIN_YEAR, MAX_YEAR = int(movies_df['year'].min()), int(movies_df['year'].max())

def create_layout():
    return dbc.Container([

        html.H2('Feel Good Films: Movie Sentiment & Box Office Revenue', style={'textAlign': 'center', 'padding': '20px 0'}),

        # Filter Card
        dbc.Card([
            dbc.CardBody(
                dbc.Row([
                    dbc.Col([
                        html.Label('Year Range', className='fw-bold small mb-1'),
                        dcc.RangeSlider(
                            id='year-slider',
                            min=MIN_YEAR, max=MAX_YEAR, step=1,
                            value=[1960, MAX_YEAR],
                            marks={y: str(y) for y in range(MIN_YEAR, MAX_YEAR + 1, 20)},
                        ),
                    ], width=8),
                    dbc.Col([
                        html.Label('Sentiment Filter', className='fw-bold small mb-1'),
                        dcc.Checklist(
                            id='sentiment-filter',
                            options=[
                                {'label': ' Positive', 'value': 'positive'},
                                {'label': ' Negative', 'value': 'negative'},
                            ],
                            value=['positive', 'negative'],
                            inline=True,
                        ),
                    ], width=4, className='d-flex flex-column justify-content-center'),
                ], align='center')
            ),
        ], style={
            'marginBottom': '20px',
            'position': 'sticky',
            'top': '0',
            'zIndex': '1000',
            'boxShadow': '0 2px 8px rgba(0,0,0,0.12)',
        }),

        # Revenue / Sentiment Stacked Area Chart
        dcc.Graph(id='area-chart'),

        # Treemap: top movies by revenue
        dcc.Graph(id='treemap-chart'),

        # Genre Tug of War + Top Keywords
        dbc.Row([
            dbc.Col(dcc.Graph(id='genre-chart'), width=6),
            dbc.Col(dcc.Graph(id='keyword-chart'), width=6),
        ])

    ], fluid=False, style={'backgroundColor': COLORS['bg'], 'padding': '20px', '--rc-slider-color': '#333'})

In [1001]:
def make_treemap(df, year_range, sentiments):
    active = sentiments or ['positive', 'negative']
    label = f'{year_range[0]}-{year_range[1]}'
    sub = df[df['sentiment_category'].isin(active)].sort_values('revenue', ascending=False)
    if sub.empty:
        return go.Figure()

    top = pd.concat([
        sub[sub['sentiment_category'] == cat].nlargest(10, 'revenue')
        for cat in active if (sub['sentiment_category'] == cat).any()
    ]).copy()
    top['sentiment_label'] = top['sentiment_category'].str.capitalize()
    top['display_title'] = top['title'].fillna('Untitled')

    fig = px.treemap(
        top,
        path=[px.Constant(label), 'sentiment_label', 'display_title'],
        values='revenue',
        color='sentiment_mean',
        color_continuous_scale=[[0, COLORS['negative']], [0.5, '#ffffff'], [1, COLORS['positive']]],
        color_continuous_midpoint=0,
        range_color=[-0.5, 0.5],
    )
    fig.update_traces(
        hovertemplate='<b>%{label}</b><br>Revenue: $%{value:,.0f}<br>Sentiment Score: %{color:.2f}<extra></extra>',
    )

    treemap_layout = {**layout_base, 'margin': dict(l=10, r=10, t=50, b=10)}
    fig.update_layout(
        title=dict(text=f'Top Movies by Revenue',
                   x=0.5, font=dict(size=18, color=COLORS['text'])),
        **treemap_layout,
        height=450,
        coloraxis_showscale=False,
    )
    return fig


In [1002]:

# Register Callbacks --------------------------------------------------------------------─
def register_callbacks(app):
    @app.callback(
        Output('area-chart', 'figure'),
        Output('treemap-chart', 'figure'),
        Output('genre-chart', 'figure'),
        Output('keyword-chart', 'figure'),
        Input('year-slider', 'value'),
        Input('sentiment-filter', 'value'),
    )
    def update_charts(year_range, sentiments):
        filtered = movies_df[movies_df['year'].between(year_range[0], year_range[1])]
        return (
            make_area_chart(filtered, sentiments=sentiments),
            make_treemap(filtered, year_range, sentiments),
            make_genre_chart(filtered, year_range),
            make_keyword_chart(filtered, year_range),
        )

In [1003]:
# Create app & run --------------------------------------------------------------------
EXT = [
    dbc.themes.LUX,
    "https://cdnjs.cloudflare.com/ajax/libs/font-awesome/6.0.0/css/all.min.css",
]

app = Dash(__name__, external_stylesheets=EXT, suppress_callback_exceptions=True)
app.title = "Movie Keyword Sentiment and Box Office Revenue"
app.layout = create_layout()
register_callbacks(app)

app.run(jupyter_mode='inline', debug=False)